# FPT Stock Price Direction Prediction - ANN

**Dataset:** FPT.csv — du lieu gia co phieu FPT (OHLCV)  
**Muc tieu:** Du doan `UpTomorrow` — ngay mai gia FPT tang (1) hay giam (0)?

Pipeline tuong tu Rain_Prediction_ANN:
1. Load & EDA
2. Feature Engineering (technical indicators + cyclic date)
3. Xu ly missing values + outliers
4. ANN: Dense(32) → Dense(32) → Dense(16) → Dropout → Dense(8) → Dropout → Dense(1)
5. Danh gia: Confusion matrix + Classification report

In [ ]:
!pip install tensorflow keras scikit-learn pandas numpy matplotlib seaborn 

   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/223.8 MB 591.3 kB/s eta 0:06:09^C
ERROR: Operation cancelled by user


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import keras
from keras.layers import Dense, Dropout
from keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from keras import callbacks
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Ready.')

# Loading The Data

In [ ]:
data = pd.read_csv('FPT.csv', parse_dates=['time'])
data = data.sort_values('time').reset_index(drop=True)

# Tinh lai return cho chac chan
data['return'] = data['close'].pct_change()

print(f'Shape: {data.shape}')
print(f'Period: {data["time"].min().date()} -> {data["time"].max().date()}')
data.head()

In [ ]:
data.info()

# Data Visualization

**Steps:**
- Phan phoi target (UpTomorrow)
- Correlation giua cac features
- Cyclic encoding ngay/thang

In [ ]:
# Target: ngay mai tang hay giam?
data['UpTomorrow'] = (data['close'].shift(-1) > data['close']).astype(int)

# Bo dong cuoi (khong co target)
data = data.dropna(subset=['return']).reset_index(drop=True)
data = data.iloc[:-1].copy()  # bo dong cuoi vi UpTomorrow = NaN

cols = ["#C2C4E2", "#EED4E5"]
sns.countplot(x=data['UpTomorrow'], palette=cols)
plt.title('Distribution of UpTomorrow (0=Down, 1=Up)')
plt.xlabel('UpTomorrow')
plt.show()

print(f'\nTy le tang (UpTomorrow=1): {data["UpTomorrow"].mean()*100:.1f}%')
print(f'So mau: {len(data)}')

# Feature Engineering

**Technical Indicators:**
- SMA 5, SMA 20 (Simple Moving Average)
- EMA 12, EMA 26 (Exponential Moving Average)
- MACD = EMA12 - EMA26
- RSI 14 (Relative Strength Index)
- Bollinger Bands (20-day, 2 std)
- Volume Ratio = Volume / SMA_Volume_20
- Momentum 5-day
- ATR 14 (Average True Range)
- Cyclic encoding: month_sin/cos, day_sin/cos

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / (loss + 1e-10)
    return 100 - (100 / (1 + rs))

def compute_atr(df, period=14):
    hl  = df['high'] - df['low']
    hpc = (df['high'] - df['close'].shift(1)).abs()
    lpc = (df['low']  - df['close'].shift(1)).abs()
    tr  = pd.concat([hl, hpc, lpc], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ── Moving Averages ──
data['sma5']  = data['close'].rolling(5).mean()
data['sma20'] = data['close'].rolling(20).mean()
data['ema12'] = data['close'].ewm(span=12).mean()
data['ema26'] = data['close'].ewm(span=26).mean()
data['macd']  = data['ema12'] - data['ema26']

# ── RSI ──
data['rsi14'] = compute_rsi(data['close'], 14)

# ── Bollinger Bands ──
bb_mid         = data['close'].rolling(20).mean()
bb_std         = data['close'].rolling(20).std()
data['bb_upper'] = bb_mid + 2 * bb_std
data['bb_lower'] = bb_mid - 2 * bb_std
data['bb_width'] = (data['bb_upper'] - data['bb_lower']) / bb_mid
data['bb_pos']   = (data['close'] - data['bb_lower']) / (data['bb_upper'] - data['bb_lower'] + 1e-10)

# ── Volume Ratio ──
data['vol_sma20'] = data['volume'].rolling(20).mean()
data['vol_ratio'] = data['volume'] / data['vol_sma20']

# ── Momentum ──
data['momentum5'] = data['close'] - data['close'].shift(5)

# ── ATR ──
data['atr14'] = compute_atr(data, 14)

# ── Price relative to MAs ──
data['close_sma5_ratio']  = data['close'] / data['sma5']  - 1
data['close_sma20_ratio'] = data['close'] / data['sma20'] - 1

# ── Cyclic Date Encoding ──
data['month'] = data['time'].dt.month
data['day']   = data['time'].dt.day

def encode_cyclic(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

data = encode_cyclic(data, 'month', 12)
data = encode_cyclic(data, 'day',   31)

print('Features computed OK.')
print(f'Shape: {data.shape}')
data[['time','close','sma20','rsi14','macd','bb_pos','UpTomorrow']].tail(5)

In [ ]:
FEATURE_COLS = [
    'return', 'vol_ratio',
    'sma5', 'sma20', 'ema12', 'ema26', 'macd',
    'rsi14',
    'bb_width', 'bb_pos',
    'momentum5', 'atr14',
    'close_sma5_ratio', 'close_sma20_ratio',
    'month_sin', 'month_cos', 'day_sin', 'day_cos',
]

corrmat = data[FEATURE_COLS + ['UpTomorrow']].corr()
cmap = sns.diverging_palette(260, -10, s=50, l=75, n=6, as_cmap=True)
plt.subplots(figsize=(14, 12))
sns.heatmap(corrmat, cmap=cmap, annot=True, fmt='.2f', square=True, annot_kws={'size': 7})
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# Data Preprocessing

- Chon features, drop NaN
- Standard Scaling
- Phat hien va loai bo outliers

In [ ]:
# Drop rows co NaN (do rolling windows)
model_df = data[FEATURE_COLS + ['UpTomorrow']].dropna().reset_index(drop=True)
print(f'Shape sau khi drop NaN: {model_df.shape}')

features = model_df[FEATURE_COLS].copy()
target   = model_df['UpTomorrow'].copy()

# Standard Scaling
s_scaler = preprocessing.StandardScaler()
features_scaled = s_scaler.fit_transform(features)
features = pd.DataFrame(features_scaled, columns=FEATURE_COLS)

features.describe().T

In [ ]:
# Phat hien outliers
colours = ["#D0DBEE", "#C2C4E2", "#EED4E5", "#D1E6DC", "#BDE2E2"]
plt.figure(figsize=(20, 6))
sns.boxenplot(data=features, palette=colours)
plt.xticks(rotation=90)
plt.title('Scaled Features - Outlier Detection')
plt.tight_layout()
plt.show()

In [ ]:
# Loai bo outliers (cac gia tri qua xa phan phoi)
features['UpTomorrow'] = target.values

features = features[features['return'].between(-3, 3)]
features = features[features['vol_ratio'].between(-3, 3)]
features = features[features['rsi14'].between(-3, 3)]
features = features[features['macd'].between(-3, 3)]
features = features[features['momentum5'].between(-3, 3)]
features = features[features['atr14'].between(-3, 3)]

print(f'Shape sau khi loai outliers: {features.shape}')

plt.figure(figsize=(20, 6))
sns.boxenplot(data=features, palette=colours)
plt.xticks(rotation=90)
plt.title('After Removing Outliers')
plt.tight_layout()
plt.show()

# Model Building - ANN

Kien truc tuong tu Rain_Prediction_ANN:

```
Input → Dense(32, relu) → Dense(32, relu) → Dense(16, relu)
      → Dropout(0.25) → Dense(8, relu) → Dropout(0.5) → Dense(1, sigmoid)
```

Optimizer: Adam(lr=0.00009)  
Loss: binary_crossentropy  
EarlyStopping: patience=20

In [ ]:
X = features.drop(['UpTomorrow'], axis=1)
y = features['UpTomorrow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Input dim: {X_train.shape[1]}')
print(f'Test - Up: {y_test.sum()} | Down: {(y_test==0).sum()}')

In [ ]:
early_stopping = callbacks.EarlyStopping(
    min_delta=0.001,
    patience=20,
    restore_best_weights=True,
)

input_dim = X_train.shape[1]

model = Sequential([
    Dense(32, kernel_initializer='uniform', activation='relu', input_dim=input_dim),
    Dense(32, kernel_initializer='uniform', activation='relu'),
    Dense(16, kernel_initializer='uniform', activation='relu'),
    Dropout(0.25),
    Dense(8,  kernel_initializer='uniform', activation='relu'),
    Dropout(0.5),
    Dense(1,  kernel_initializer='uniform', activation='sigmoid'),
])

opt = Adam(learning_rate=0.00009)
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=150,
    callbacks=[early_stopping],
    validation_split=0.2,
    verbose=1,
)

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df['loss'],     '#BDE2E2', label='Training loss')
axes[0].plot(history_df['val_loss'], '#C2C4E2', label='Validation loss')
axes[0].set_title('Training and Validation Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df['accuracy'],     '#BDE2E2', label='Training accuracy')
axes[1].plot(history_df['val_accuracy'], '#C2C4E2', label='Validation accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

# Conclusion

- Confusion Matrix
- Classification Report

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

In [ ]:
# Confusion Matrix
cmap1 = sns.diverging_palette(260, -10, s=50, l=75, n=5, as_cmap=True)
plt.subplots(figsize=(8, 6))
cf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(cf_matrix / cf_matrix.sum(), cmap=cmap1, annot=True,
            annot_kws={'size': 14}, fmt='.3f',
            xticklabels=['Down (0)', 'Up (1)'],
            yticklabels=['Down (0)', 'Up (1)'])
plt.title('Confusion Matrix - FPT UpTomorrow Prediction')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))